# 머신러닝 기반 텍스트 분류
1. 데이터 준비 : 파일 로딩, 입력데이터, 출력데이터, 학습/테스트 데이터 분리, 특징추출
2. 학습-평가
3. 배포 준비

### 1. 데이터 준비

In [1]:
import pandas as pd
datafile = './data/Korean_movie_reviews_2016.csv' #label이 있는 자료
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


### 1-1. 입력 데이터와 정답 데이터로 추출

In [3]:
review_list = list(data_df.review)     #입력 데이터
label_list = list(data_df.label)       #출력 데이터
len(review_list), len(label_list)

(165384, 165384)

### 1-2. 데이터 분리(학습용, 테스트용)

In [4]:
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(test_X), len(train_y), len(test_y)

(148845, 16539, 148845, 16539)

### 1-3. 특징 추출

In [5]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    tokenizer = Okt().pos
    return [word for word, tag in tokenizer(text) if tag in my_tags and word not in my_stopwords]

In [81]:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(tokenizer=korean_tokenizer, max_features=1000)
vectorizer.fit(train_X)


c:\Users\user\AppData\Local\anaconda3\envs\textmine26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TfidfVectorizer(max_features=1000,
                tokenizer=<function korean_tokenizer at 0x000001B1E116DCA0>)

In [82]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가', '가고', '가는', '가볍', '가서', '가슴', '가장', '가족', '가지', '각본'],
       dtype=object))

In [83]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [84]:
# 테스트 데이터 특징 추출(나중에 해야 한다)
test_X_fv = vectorizer.transform(test_X)

In [85]:
print(train_X_fv)

  (np.int32(0), np.int32(677))	0.6302054632707322
  (np.int32(0), np.int32(821))	0.5327437551555895
  (np.int32(0), np.int32(954))	0.5648231275421035
  (np.int32(1), np.int32(106))	0.5612253445167878
  (np.int32(1), np.int32(368))	0.3419982783632707
  (np.int32(1), np.int32(542))	0.3075419403588574
  (np.int32(1), np.int32(625))	0.2518407052666464
  (np.int32(1), np.int32(693))	0.3354675779725525
  (np.int32(1), np.int32(696))	0.32111262392140233
  (np.int32(1), np.int32(822))	0.29883692125894046
  (np.int32(1), np.int32(955))	0.3241946725961087
  (np.int32(2), np.int32(93))	0.3737146348182635
  (np.int32(2), np.int32(216))	0.23482024411005098
  (np.int32(2), np.int32(273))	0.3241394088188063
  (np.int32(2), np.int32(293))	0.24395337431478717
  (np.int32(2), np.int32(388))	0.2946330613239405
  (np.int32(2), np.int32(514))	0.3271383964037372
  (np.int32(2), np.int32(529))	0.3502915064120044
  (np.int32(2), np.int32(587))	0.2742682278614795
  (np.int32(2), np.int32(661))	0.30967925682341

In [86]:
# 정답 데이터 ndarray로 변환 : sklearn은 정답 데이터를 ndarray로 받기를 원한다 
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
train_y[:10]

array([0, 1, 0, 0, 1, 1, 1, 1, 0, 1])

# 2. 머신 러닝 - 모델 학습
1. 의사결정트리, Decision Tree
2. 랜덤포레스트, RandomForest
3. 나이즈 베이즈 분류, Navie bayes Classfier
4. 로지스틱 회귀, Logistic Regression
5. SVM, Support Vector Machine
6. 퍼셉트론, Perceptron

In [87]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [88]:
def getScores(model, train_x, train_y, test_x, test_y):
    train_score = model.score(train_x, train_y) * 100
    test_score = model.score(test_x, test_y) * 100
    return train_score, test_score

## 1. Decision Tree

In [89]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv,train_y) # 학습 시키기 

DecisionTreeClassifier()

In [90]:
train_score, test_score = getScores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

98.11548926735865 80.1559949210956


In [91]:
score_df.loc['DecisionTree'] = [train_score, test_score]  # 행 지정해주기 위해서 loc사용(loc없으면 column으로 인식)
score_df

,train,test
DecisionTree,98.115489,80.155995


## 2. Random Forest

In [92]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [93]:
train_score, test_score = getScores(rf, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score]  
score_df


98.1141455876919 84.8418888687345


,train,test
DecisionTree,98.115489,80.155995
RandomForest,98.114146,84.841889


## 3. Naive Bayes Classfier

In [94]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [95]:
train_score, test_score = getScores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['NavieBayes'] = [train_score, test_score]  
score_df

85.26252141489469 85.48884454924723


,train,test
DecisionTree,98.115489,80.155995
RandomForest,98.114146,84.841889
NavieBayes,85.262521,85.488845


## 4. 로지스틱 회귀분석

In [96]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [97]:
train_score, test_score = getScores(lr, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['LogisticRegression'] = [train_score, test_score]  
score_df

86.12919479995969 86.23858758086946


,train,test
DecisionTree,98.115489,80.155995
RandomForest,98.114146,84.841889
NavieBayes,85.262521,85.488845
LogisticRegression,86.129195,86.238588


## 5. 서포트백터머신(SVM)

In [98]:
from sklearn.svm import LinearSVC

lvc = LinearSVC()
lvc.fit(train_X_fv, train_y)

LinearSVC()

In [99]:
train_score, test_score = getScores(lvc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score]  
score_df

86.10232120662434 86.22649495132717


,train,test
DecisionTree,98.115489,80.155995
RandomForest,98.114146,84.841889
NavieBayes,85.262521,85.488845
LogisticRegression,86.129195,86.238588
SVM,86.102321,86.226495


In [100]:
score_df.sort_values(by='test', ascending=False)  # ascending = 내림차순

,train,test
LogisticRegression,86.129195,86.238588
SVM,86.102321,86.226495
NavieBayes,85.262521,85.488845
RandomForest,98.114146,84.841889
DecisionTree,98.115489,80.155995


### 6. 배포 준비
- 기능 구현
- 모델 저장

In [101]:
# review = '영화가 너무 재미있다'
review = '영화가 재미없다'

def analyze_sentiment(review):
    # 전처리 및 특징 벡터 추출(feature)
    review_fv = vectorizer.transform([review])
    #print(review_fv)

    result = rf.predict(review_fv)
    #print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'

    return show

show = analyze_sentiment(review)
print(f'{review} -> {show}')

영화가 재미없다 -> 부정


In [102]:
reviews = ['영화가 너무 재미있다', '이게 영화냐? 나도 만들겠다.', '개꿀잼', '핵노잼']

for review in reviews:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다 -> 긍정
이게 영화냐? 나도 만들겠다. -> 부정
개꿀잼 -> 긍정
핵노잼 -> 부정


In [103]:
import joblib

vectorizer_file = './model/sa_movie_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_file)

model_file = './model/sa_movie_model.pkl'
joblib.dump(rf, model_file)

['./model/sa_movie_model.pkl']